In [ ]:
# =============================================================================
# PORTFOLIO | CUSTOMER MIGRATION ANALYSIS PIPELINE
# =============================================================================
# Purpose:
# Build an analytical base to track customers migrated between segments,
# evaluating behavior by cohort, products, income, churn, rating,
# primary relationship (principality), and financial indicators.
#
# Note:
# This code has been anonymized for portfolio purposes.
# All server names, credentials, schemas, tables, and users have been
# replaced with fictitious values.
#
# Domain terms kept as-is (common in financial data pipelines):
# - ANOMES: YearMonth key in the format YYYYMM
# - PDD: Allowance for loan losses (provision for doubtful debts)
# - IPP: Product Penetration Index (product count / client count)
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

from functools import reduce
from datetime import datetime
from dateutil.relativedelta import relativedelta

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, DecimalType, TimestampType
from pyspark.storagelevel import StorageLevel
from pyspark.sql.utils import AnalysisException

# Dummy client for Oracle connection
from oracle_to_bricks import OracleClient

import os
import sys


# =============================================================================
# ANONYMIZED CONFIGURATION
# =============================================================================

ORACLE_HOST = "oracle-host-dummy.company.local"
ORACLE_PORT = 1521
ORACLE_SID = "oracle_service_dummy"
ORACLE_USER = "dummy_user"
ORACLE_PASSWORD = "dummy_password"

CATALOG_PRD = "dummy_prod_catalog"
SCHEMA_ANALYTICS = "dummy_analytics_schema"
SCHEMA_CLIENTS = "dummy_clients_schema"
SCHEMA_AUXILIARY = "dummy_auxiliary_schema"
SCHEMA_ORACLE = "DUMMY_ORACLE_SCHEMA"

TABLE_MIGRATED_CLIENTS = f"{CATALOG_PRD}.{SCHEMA_CLIENTS}.tb_migrated_clients_dummy"
TABLE_CLIENT_RESULTS = f"{CATALOG_PRD}.{SCHEMA_ANALYTICS}.tb_client_results_dummy"
TABLE_PRINCIPALITY = f"{CATALOG_PRD}.{SCHEMA_ANALYTICS}.tb_principality_dummy"
TABLE_CHURN_MAPPING = f"{CATALOG_PRD}.{SCHEMA_AUXILIARY}.churn_mapping_dummy"
TABLE_PACKAGE = f"{CATALOG_PRD}.{SCHEMA_AUXILIARY}.tb_package_dummy"
TABLE_CONTROL_GROUP = f"{CATALOG_PRD}.{SCHEMA_AUXILIARY}.tb_control_group_dummy"

TABLE_CLIENT_INCOME = f"{CATALOG_PRD}.{SCHEMA_AUXILIARY}.tb_client_income_dummy"
TABLE_CLIENT_CHURN = f"{CATALOG_PRD}.{SCHEMA_AUXILIARY}.tb_client_churn_dummy"
TABLE_PREAPPROVED_MONO = f"{CATALOG_PRD}.{SCHEMA_AUXILIARY}.tb_preapproved_mono_dummy"
TABLE_PREAPPROVED_ACCOUNT_HOLDER = f"{CATALOG_PRD}.{SCHEMA_AUXILIARY}.tb_preapproved_account_holder_dummy"

TABLE_DESTINATION_MIGRATED = f"{CATALOG_PRD}.{SCHEMA_CLIENTS}.tb_migrated_cohort_validation_dummy"
TABLE_DESTINATION_COMPARATIVE = f"{CATALOG_PRD}.{SCHEMA_CLIENTS}.tb_migrated_cohort_comparative_dummy"

ORACLE_RATING_PARTITION = f"{SCHEMA_ORACLE}.TB_CLIENT_RATING_DUMMY"


# =============================================================================
# ANONYMIZED ORACLE CONNECTION
# =============================================================================

client = OracleClient(ORACLE_USER, ORACLE_PASSWORD)
client.help_check()


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def get_previous_anomes(anomes: int) -> int:
    """
    Calculates the previous month in YYYYMM format.
    """
    year = anomes // 100
    month = anomes % 100

    if month == 1:
        return (year - 1) * 100 + 12

    return year * 100 + (month - 1)


def read_table(table: str) -> DataFrame:
    """
    Reads a Delta/Spark table.
    """
    return spark.table(table)


def get_max_date(table: str, date_column: str, anomes: int):
    """
    Retrieves the maximum available date within a specific month.
    """
    query = (
            f"SELECT MAX({date_column}) "
            f"FROM {table} "
            f"WHERE CONCAT(SUBSTR({date_column}, 1, 4), SUBSTR({date_column}, 6, 2)) = '{anomes}'"
            )

    return spark.sql(query).collect()[0][0]


def get_cohorts(initial_anomes: int, positive_months: int, negative_months: int) -> dict:
    """
    Returns a dictionary with cohorts relative to the initial month.
    Example:
    m0, m+1, m+2, m-1, m-2, etc.
    """
    dt = datetime.strptime(str(initial_anomes), "%Y%m")

    cohorts_dict = {"m0": int(dt.strftime("%Y%m"))}

    cohorts_dict.update({
        f"m-{i}": int((dt - relativedelta(months=i)).strftime("%Y%m"))
        for i in range(1, negative_months)
    })

    cohorts_dict.update({
        f"m+{i}": int((dt + relativedelta(months=i)).strftime("%Y%m"))
        for i in range(1, positive_months + 1)
    })

    return cohorts_dict


def create_income_bracket(income_column):
    """
    Creates the income bracket classification.
    """
    return (
        F.when((income_column.isNull()) | (income_column <= 2000), "A. 0K to 2K")
         .when((income_column > 2000) & (income_column <= 4000), "B. 2K to 4K")
         .when((income_column > 4000) & (income_column <= 7000), "C. 4K to 7K")
         .when((income_column > 7000) & (income_column <= 10000), "D. 7K to 10K")
         .when((income_column > 10000) & (income_column <= 15000), "E. 10K to 15K")
         .when((income_column > 15000) & (income_column <= 20000), "F. 15K to 20K")
         .when((income_column > 20000) & (income_column <= 25000), "G. 20K to 25K")
         .when((income_column > 25000) & (income_column <= 30000), "H. 25K to 30K")
         .when((income_column > 30000) & (income_column <= 35000), "I. 30K to 35K")
         .when((income_column > 35000) & (income_column <= 50000), "J. 35K to 50K")
         .when(income_column > 50000, "K. Above 50K")
    )


def create_client_cluster(status_column, account_activity_column):
    """
    Classifies the client according to status and account activity.
    """
    return (
        F.when((status_column == "ACCOUNT_HOLDER") & (account_activity_column == "ACTIVE"), "ACCOUNT_HOLDER_ACTIVE")
         .when((status_column == "ACCOUNT_HOLDER") & (account_activity_column == "INACTIVE"), "ACCOUNT_HOLDER_INACTIVE")
         .when((status_column == "MONO_PRODUCT") & (account_activity_column == "INACTIVE"), "MONO_PRODUCT_INACTIVE")
         .when((status_column == "MONO_PRODUCT") & (account_activity_column == "ACTIVE"), "MONO_PRODUCT_ACTIVE")
         .otherwise(None)
    )


# =============================================================================
# STUDY PARAMETERS
# =============================================================================

clusters = ["UPGRADE", "DOWNGRADE", "SPECIAL_SEGMENTATION_ADJUSTMENT"]

migration_anomes = 202601

original_cohort_months = [
                        (202508, "-5"),
                        (202509, "-4"),
                        (202510, "-3"),
                        (202511, "-2"),
                        (202512, "-1"),
                        (202601, "0"),
                        (202602, "1"),
                        (202603, "2"),
                        (202604, "3"),
                        (202605, "4"),
                        (202606, "5")
                    ]

min_cohort = original_cohort_months[0][0]
max_cohort = original_cohort_months[-1][0]

# Capturing months prior to time 0
pre_migration_cohorts = [month for month, cohort in original_cohort_months if cohort.startswith("-")]


# =============================================================================
# MIGRATED CLIENTS BASE
# =============================================================================

df_migrated = (spark.table(TABLE_MIGRATED_CLIENTS)
                .filter(
                        (F.col("ADJUSTMENT_REASON_FLAG").isin(clusters)) &
                        (F.col("Segmentation") == "YES") &
                        (F.col("ANOMES") == migration_anomes)).persist(StorageLevel.MEMORY_AND_DISK))

# =============================================================================
# CHECKS ALREADY PROCESSED MONTHS
# =============================================================================

try:
    processed_months = (spark.table(TABLE_DESTINATION_MIGRATED)
                            .select("ANOMES_REF")
                            .distinct()
                            .rdd
                            .flatMap(lambda x: x)
                            .collect())
    destination_table_exists = True

except Exception:
    print(f"Destination table does not exist yet or could not be read: {TABLE_DESTINATION_MIGRATED}")
    print("Processing will continue for all defined months.")

    processed_months = []
    destination_table_exists = False


cohort_months = [(month, cohort) for month, cohort in original_cohort_months if month not in processed_months]

if len(cohort_months) == 0:
    print("No new ANOMES_REF to process.")
    dbutils.notebook.exit("No new ANOMES_REF to process.")

print(f"Months to be processed: {[m for m, c in cohort_months]}")


# =============================================================================
# AUXILIARY BASES
# =============================================================================

analytical_base = (read_table(TABLE_CLIENT_RESULTS).filter(F.col("ANOMES").between(min_cohort, max_cohort)).persist(StorageLevel.MEMORY_AND_DISK))

principality = (read_table(TABLE_PRINCIPALITY).withColumn("ANOMES", F.col("ANOMES").cast("int")))

churn_mapping = read_table(TABLE_CHURN_MAPPING)

df_package = (read_table(TABLE_PACKAGE).persist(StorageLevel.MEMORY_AND_DISK))

product_columns = [c for c in analytical_base.columns if c.startswith("QT_")]

# =============================================================================
# MAIN LOOP | MIGRATED CLIENTS
# =============================================================================

dfs = []

for month, cohort in cohort_months:

    print(f"Processing {month}")

    destination_cohort = str(int(cohort) + 1)

    analytical_base_month = analytical_base.filter(F.col("ANOMES") == month)
    principality_month = principality.filter(F.col("ANOMES") == month)

    print("CHECKING INCOME")

    max_income_date = get_max_date(TABLE_CLIENT_INCOME, "load_ref_date", month)

    df_income_base = spark.sql(f"""
        SELECT
            CAST(CLIENT_CODE AS BIGINT) AS CLIENT_ID,
            MAX(CAST(INCOME_VALUE AS DECIMAL(18, 0))) AS RISK_INCOME
        FROM {TABLE_CLIENT_INCOME}
        WHERE load_ref_date = '{max_income_date}'
        GROUP BY CAST(CLIENT_CODE AS BIGINT) """)

    df_income = (df_income_base.withColumn("RISK_INCOME_BRACKET", create_income_bracket(F.col("RISK_INCOME"))))

    print("CHECKING CHURN")

    max_churn_date = get_max_date(TABLE_CLIENT_CHURN, "load_ref_date", month)

    df_churn = (read_table(TABLE_CLIENT_CHURN).alias("A")
                .filter(F.col("load_ref_date") == max_churn_date)
                .join(
                    F.broadcast(churn_mapping.alias("B")),
                    F.col("A.churn_code").cast("int") == F.col("B.CHURN_CODE").cast("int"),
                    how="left")
                .select(
                    F.col("A.PERSON_ID"),
                    F.col("B.CHURN_CODE"),
                    F.col("B.STATUS")))

    print("CHECKING PRE-APPROVED CREDIT TO MONO PRODUCT CLIENTS")

    max_preapproved_mono_date = get_max_date(TABLE_PREAPPROVED_MONO, "load_ref_date", month)

    df_preapproved_mono = (read_table(TABLE_PREAPPROVED_MONO).filter(F.col("load_ref_date") == max_preapproved_mono_date))

    print("CHECKING PRE-APPROVED CREDIT TO ACCOUNT HOLDERS")

    max_preapproved_account_date = get_max_date(TABLE_PREAPPROVED_ACCOUNT_HOLDER, "load_ref_date", month)

    df_preapproved_account = (read_table(TABLE_PREAPPROVED_ACCOUNT_HOLDER).filter(F.col("load_ref_date") == max_preapproved_account_date))

    print("CHECKING RATING")

    rating_query = f"""SELECT PERSON_ID, RATING_M0 FROM {ORACLE_RATING_PARTITION} PARTITION(P_{month})"""

    df_rating = client.fetch_data_with_query(rating_query)

    df_package_month = df_package.filter(F.col("ANOMES") == month)

    print("JOINING BASES")

    selection = (df_migrated.alias("MIGR")
        .join(analytical_base_month.alias("BA"), F.col("MIGR.CLIENT_ID") == F.col("BA.CLIENT_ID"), how="left")
        .join(df_package_month.alias("PKG"), F.col("PKG.CLIENT_ID") == F.col("MIGR.CLIENT_ID"), how="left")
        .join(df_income.alias("INC"), F.col("INC.CLIENT_ID") == F.col("MIGR.CLIENT_ID"), how="left")
        .join(df_rating.alias("RATING"), F.col("RATING.PERSON_ID") == F.col("MIGR.PERSON_ID"), how="left")
        .join(df_churn.alias("CHURN"), F.col("CHURN.PERSON_ID") == F.col("MIGR.PERSON_ID"), how="left")
        .join(df_preapproved_mono.alias("PAP_MONO"), F.col("MIGR.CLIENT_ID") == F.col("PAP_MONO.CLIENT_ID"), how="left")
        .join(df_preapproved_account.alias("PAP_ACC"), F.col("MIGR.CLIENT_ID") == F.col("PAP_ACC.CLIENT_ID"), how="left")
        .join(principality_month.alias("PRINC"), F.col("MIGR.CLIENT_ID") == F.col("PRINC.CLIENT_ID"), how="left")

        .select(
            F.lit(month).alias("ANOMES_REF"),
            F.concat(F.lit("M"), F.lit(cohort)).alias("COHORT"),
            F.concat(F.lit("M"), F.lit(destination_cohort)).alias("DESTINATION_COHORT"),

            F.col("MIGR.ADJUSTMENT_REASON_FLAG"),
            F.col("MIGR.CLIENT_ID").alias("MIGRATED_CLIENT_ID"),
            F.col("BA.CLIENT_ID").alias("RESULT_CLIENT_ID"),

            F.col("MIGR.Segmentation").alias("SEGMENTATION_FLAG"),
            F.col("MIGR.origin_segment_code").alias("ORIGIN_SEGMENT_CODE"),
            F.col("MIGR.target_segment_code").alias("TARGET_SEGMENT_CODE"),
            F.col("BA.SEGMENT_CODE").alias("BA_SEGMENT_CODE"),

            F.col("BA.CLIENT_STATUS_FLAG"),
            F.col("CHURN.STATUS").alias("CHURN_STATUS"),
            F.col("BA.ACCOUNT_ACTIVITY_FLAG"),

            F.col("PKG.ANOMES").alias("PACKAGE_ANOMES"),
            F.col("PKG.package_type"),
            F.col("PKG.package_description"),

            F.col("PRINC.PRINCIPALITY"),
            F.col("PRINC.PIX_STRONG_KEY"),
            F.col("PRINC.PIX_WEAK_KEY"),

            F.col("INC.RISK_INCOME_BRACKET").alias("INCOME_BRACKET"),
            F.col("RATING.RATING_M0").alias("RATING"),

            F.col("BA.OPERATING_MARGIN").alias("MOB"),
            F.col("BA.NET_OPERATING_MARGIN").alias("MOL"),
            F.col("BA.PDD"),
            F.col("BA.ASSET_BALANCE"),
            F.col("BA.LIABILITY_BALANCE"),

            F.when(
                (F.col("BA.CLIENT_STATUS_FLAG") == "MONO_PRODUCT") &
                (F.col("PAP_MONO.CARD_LIMIT_VALUE") > 0) &
                (F.col("PAP_MONO.ADDITIONAL_LIMIT_VALUE") > 0),
                F.lit(1)
            )
            .when(
                (F.col("BA.CLIENT_STATUS_FLAG") == "ACCOUNT_HOLDER") &
                (F.col("PAP_ACC.CARD_LIMIT_VALUE") > 0),
                F.lit(1)
            ).otherwise(F.lit(0)).alias("PREAPPROVED_CARD_LIMIT_FLAG"),

            F.when(F.col("BA.IMPLEMENTED_CARD_LIMIT_VALUE") > 0, F.lit(1)).otherwise(F.lit(0)).alias("IMPLEMENTED_LIMIT_FLAG"),

            *[F.col(f"BA.{field}").alias(field) for field in product_columns]
        ).dropDuplicates())

    intermediate_df = (selection
                        .select(
                            "*",
                
                            F.when(
                                (F.col("TARGET_SEGMENT_CODE") == F.col("BA_SEGMENT_CODE")) |
                                (F.col("ANOMES_REF").isin(pre_migration_cohorts)),
                                F.lit("0")
                            )
                            .otherwise(F.lit("1"))
                            .alias("LEFT_RESTRICTED_SEGMENT_FLAG"),
                
                            create_client_cluster(
                                F.col("CLIENT_STATUS_FLAG"),
                                F.col("ACCOUNT_ACTIVITY_FLAG")
                            )
                            .alias("CLIENT_CLUSTER"),
                
                            F.when(F.col("package_description") == "Dummy Premium Package", F.lit(1))
                             .otherwise(F.lit(0))
                             .alias("SELECTED_PACKAGE_FLAG")))

    print(f"STACKING {month}")
    dfs.append(intermediate_df)


print("-------------------------------------------------------------------------------")
print("OK! Loop finished")


# =============================================================================
# FINAL UNION
# =============================================================================

df_final = (
    reduce(DataFrame.unionByName, dfs)
    .dropDuplicates()
)


# =============================================================================
# CLIENT MOVEMENT
# =============================================================================

full_window = (
    Window
    .partitionBy("RESULT_CLIENT_ID")
    .orderBy("ANOMES_REF")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

df_movement_1 = (
    df_final
    .withColumn("INITIAL_ANOMES", F.first("ANOMES_REF").over(full_window))
    .withColumn("FINAL_ANOMES", F.last("ANOMES_REF").over(full_window))
    .withColumn("INITIAL_CLUSTER", F.first("CLIENT_CLUSTER").over(full_window))
    .withColumn("FINAL_CLUSTER", F.last("CLIENT_CLUSTER").over(full_window))
)

order_window = Window.partitionBy("RESULT_CLIENT_ID").orderBy("ANOMES_REF")

df_movement_2 = (
    df_movement_1
    .withColumn("DESTINATION_CLUSTER", F.lead("CLIENT_CLUSTER").over(order_window))
    .withColumn("DESTINATION_ANOMES", F.lead("ANOMES_REF").over(order_window))
)


# =============================================================================
# FINAL AGGREGATION | MIGRATED CLIENTS
# =============================================================================

grouping_columns = ["ANOMES_REF",
                    "DESTINATION_ANOMES",
                    "INITIAL_ANOMES",
                    "FINAL_ANOMES",
                    "INITIAL_CLUSTER",
                    "FINAL_CLUSTER",
                    "COHORT",
                    "DESTINATION_COHORT",
                    "SEGMENTATION_FLAG",
                    "CLIENT_STATUS_FLAG",
                    "CHURN_STATUS",
                    "ACCOUNT_ACTIVITY_FLAG",
                    "ADJUSTMENT_REASON_FLAG",
                    "RATING",
                    "ORIGIN_SEGMENT_CODE",
                    "TARGET_SEGMENT_CODE",
                    "BA_SEGMENT_CODE",
                    "LEFT_RESTRICTED_SEGMENT_FLAG",
                    "SELECTED_PACKAGE_FLAG",
                    "CLIENT_CLUSTER",
                    "DESTINATION_CLUSTER",
                    "INCOME_BRACKET",
                    "PREAPPROVED_CARD_LIMIT_FLAG",
                    "IMPLEMENTED_LIMIT_FLAG",
                    "PRINCIPALITY",
                    "PIX_STRONG_KEY",
                    "PIX_WEAK_KEY"]

df_client = (df_movement_2
                .groupBy(*grouping_columns, "RESULT_CLIENT_ID")
                .agg(*[F.max(c).alias(c) for c in product_columns],
                     F.sum("MOB").alias("MOB"),
                     F.sum("MOL").alias("MOL"),
                     F.sum("PDD").alias("PDD"),
                     F.sum("ASSET_BALANCE").alias("ASSET_BALANCE"),
                     F.sum("LIABILITY_BALANCE").alias("LIABILITY_BALANCE")))

aggregation_expressions = (
                [F.countDistinct("RESULT_CLIENT_ID").alias("CLIENT_COUNT")] +
                [
                    F.sum("MOB").alias("MOB"),
                    F.sum("MOL").alias("MOL"),
                    F.sum("PDD").alias("PDD"),
                    F.sum("ASSET_BALANCE").alias("ASSET_BALANCE"),
                    F.sum("LIABILITY_BALANCE").alias("LIABILITY_BALANCE")] +
                [F.sum(c).alias(c) for c in product_columns])

df_aggregated = (df_client.groupBy(*grouping_columns).agg(*aggregation_expressions))

ipp_columns = [(F.col(c) / F.col("CLIENT_COUNT")).alias(f"IPP_{c}") for c in product_columns]

df_final_result = df_aggregated.select("*", *ipp_columns)


# =============================================================================
# FINAL WRITE | MIGRATED CLIENTS
# =============================================================================

print("SAVING")

if destination_table_exists:
    print(f"Appending to existing table: {TABLE_DESTINATION_MIGRATED}")

    df_final_result.write.mode("append").saveAsTable(TABLE_DESTINATION_MIGRATED)

else:
    print(f"Creating destination table: {TABLE_DESTINATION_MIGRATED}")

    df_final_result.write.mode("overwrite").saveAsTable(TABLE_DESTINATION_MIGRATED)

print("MIGRATED CLIENTS PROCESS COMPLETED SUCCESSFULLY.")


# =============================================================================
# CACHE CLEANUP
# =============================================================================

df_migrated.unpersist()
analytical_base.unpersist()
df_package.unpersist()


# =============================================================================
# STAGE 2 | COMPARISON WITH CONTROL GROUP
# =============================================================================

periods = get_cohorts(migration_anomes, 5, 7)

valid_periods = [periods["m0"],
                 periods["m+1"],
                 periods["m+2"],
                 periods["m+3"],
                 periods["m+4"],
                 periods["m+5"],
                 periods["m-1"],
                 periods["m-2"],
                 periods["m-3"],
                 periods["m-4"],
                 periods["m-5"]]

min_anomes = min(valid_periods)
max_anomes = max(valid_periods)

loop_months = sorted(set(valid_periods))


# =============================================================================
# CHECKS EXISTING MONTHS IN THE COMPARATIVE TABLE
# =============================================================================

try:
    existing_anomes = (
        spark.table(TABLE_DESTINATION_COMPARATIVE)
        .select("ANOMES")
        .distinct()
        .rdd
        .flatMap(lambda x: x)
        .collect())

    print(f"ANOMES already present in the table: {sorted(existing_anomes)}")

except AnalysisException:
    print("Comparative table does not exist yet. All months will be processed.")
    existing_anomes = []


months_to_process = [m for m in loop_months if m not in existing_anomes]

print(f"Months to process: {months_to_process}")


# =============================================================================
# RELOADING BASES FOR THE COMPARATIVE STAGE
# =============================================================================

df_migrated = (
    spark.table(TABLE_MIGRATED_CLIENTS)
    .filter(
        (F.col("ADJUSTMENT_REASON_FLAG").isin(clusters)) &
        (F.col("Segmentation") == "YES") &
        (F.col("ANOMES") == migration_anomes))
    .persist(StorageLevel.MEMORY_AND_DISK))

analytical_base = (
    read_table(TABLE_CLIENT_RESULTS)
    .filter(F.col("ANOMES") >= min_anomes)
    .persist(StorageLevel.MEMORY_AND_DISK))

df_control_group = (
    read_table(TABLE_CONTROL_GROUP)
    .persist(StorageLevel.MEMORY_AND_DISK))

principality = (
    read_table(TABLE_PRINCIPALITY)
    .withColumn("ANOMES", F.col("ANOMES").cast("int"))
    .filter(F.col("ANOMES").isin(loop_months))
    .persist(StorageLevel.MEMORY_AND_DISK))

product_columns = [c for c in analytical_base.columns if c.startswith("QT_")]


# =============================================================================
# COMPARATIVE LOOP
# =============================================================================

dfs = []

for month in months_to_process:

    print(f"PROCESSING MONTH {month}")

    print("CHECKING INCOME")

    max_income_date = get_max_date(TABLE_CLIENT_INCOME,"load_ref_date",month)

    df_income_base = spark.sql(f"""
        SELECT
            CAST(CLIENT_CODE AS BIGINT) AS CLIENT_ID,
            MAX(CAST(INCOME_VALUE AS DECIMAL(18, 0))) AS RISK_INCOME
        FROM {TABLE_CLIENT_INCOME}
        WHERE load_ref_date = '{max_income_date}'
        GROUP BY CAST(CLIENT_CODE AS BIGINT)
    """)

    df_income = (df_income_base.withColumn("RISK_INCOME_BRACKET",create_income_bracket(F.col("RISK_INCOME"))))

    print("CHECKING RATING")

    rating_query = f"""
        SELECT PERSON_ID, RATING_M0
        FROM {ORACLE_RATING_PARTITION} PARTITION(P_{month})
    """

    df_rating = client.fetch_data_with_query(rating_query)

    print("CHECKING PRE-APPROVED MONO CLIENTS")

    max_preapproved_mono_date = get_max_date(TABLE_PREAPPROVED_MONO,"load_ref_date",month)

    df_preapproved_mono = (read_table(TABLE_PREAPPROVED_MONO).filter((F.col("load_ref_date") == max_preapproved_mono_date) &
                                                                    (F.col("CARD_LIMIT_VALUE") > 0) &
                                                                    (F.col("ADDITIONAL_LIMIT_VALUE") > 0))
        .select(F.col("CLIENT_ID").alias("PAP_MONO_CLIENT_ID"),
                "CARD_LIMIT_VALUE",
                "ADDITIONAL_LIMIT_VALUE").dropDuplicates(["PAP_MONO_CLIENT_ID"]))

    print("CHECKING PRE-APPROVED ACCOUNT HOLDERS")

    max_preapproved_account_date = get_max_date(TABLE_PREAPPROVED_ACCOUNT_HOLDER,"load_ref_date",month)

    df_preapproved_account = (read_table(TABLE_PREAPPROVED_ACCOUNT_HOLDER).filter((F.col("load_ref_date") == max_preapproved_account_date) &
                                                                                  (F.col("CARD_LIMIT_VALUE") > 0))
        .select(F.col("CLIENT_ID").alias("PAP_ACC_CLIENT_ID"),
                F.col("CARD_LIMIT_VALUE").alias("ACC_CARD_LIMIT_VALUE")).dropDuplicates(["PAP_ACC_CLIENT_ID"]))

    base_month = analytical_base.filter(F.col("ANOMES") == month)
    principality_month = principality.filter(F.col("ANOMES") == month)

    comparative_result = (
        base_month.alias("A")
        .join(
            df_preapproved_mono.alias("PAP_MONO"),
            F.col("A.CLIENT_ID") == F.col("PAP_MONO.PAP_MONO_CLIENT_ID"),
            how="left")
        .join(
            df_preapproved_account.alias("PAP_ACC"),
            F.col("A.CLIENT_ID") == F.col("PAP_ACC.PAP_ACC_CLIENT_ID"),
            how="left")
        .join(
            df_income.alias("INC"),
            F.col("A.CLIENT_ID") == F.col("INC.CLIENT_ID"),
            how="left")
        .join(
            df_rating.alias("RATING"),
            F.col("A.PERSON_ID") == F.col("RATING.PERSON_ID"),
            how="left")
        .join(
            df_control_group.alias("CTRL"),
            F.col("A.CLIENT_ID") == F.col("CTRL.CLIENT_ID"),
            how="left")
        .join(
            principality_month.alias("PRINC"),
            F.col("A.CLIENT_ID") == F.col("PRINC.CLIENT_ID"),
            how="left")
        .select(
            F.col("A.ANOMES"),
            F.col("A.SEGMENT_CODE").alias("SEGMENT_CODE"),
            F.col("A.CLIENT_ID").alias("RESULT_CLIENT_ID"),
            F.col("A.ACCOUNT_ACTIVITY_FLAG"),
            F.col("A.CLIENT_STATUS_FLAG"),

            F.col("INC.RISK_INCOME_BRACKET"),
            F.col("RATING.RATING_M0").alias("RATING"),

            F.col("PRINC.PIX_WEAK_KEY"),
            F.col("PRINC.PRINCIPALITY"),
            F.col("PRINC.PIX_STRONG_KEY"),

            F.when(
                (F.col("A.CLIENT_STATUS_FLAG") == "MONO") &
                (F.col("PAP_MONO.CARD_LIMIT_VALUE") > 0) &
                (F.col("PAP_MONO.ADDITIONAL_LIMIT_VALUE") > 0),
                F.lit(1))
            .when(
                (F.col("A.CLIENT_STATUS_FLAG") == "ACCOUNT_HOLDER") &
                (F.col("PAP_ACC.ACC_CARD_LIMIT_VALUE") > 0),
                F.lit(1))
            .otherwise(F.lit(0)).alias("PREAPPROVED_CARD_LIMIT_FLAG"),

            F.when(
                F.col("A.IMPLEMENTED_CARD_LIMIT_VALUE") > 0,
                F.lit(1)).otherwise(F.lit(0)).alias("IMPLEMENTED_LIMIT_FLAG"),

            create_client_cluster(F.col("A.CLIENT_STATUS_FLAG"),F.col("A.ACCOUNT_ACTIVITY_FLAG"))
                .alias("CLIENT_CLUSTER"),

            F.when(F.col("CTRL.CLIENT_ID").isNotNull(), F.lit(1)).otherwise(F.lit(0)).alias("CONTROL_GROUP_FLAG"),

            F.col("A.OPERATING_MARGIN").alias("MOB"),
            F.col("A.NET_OPERATING_MARGIN").alias("MOL"),
            F.col("A.PDD"),
            F.col("A.ASSET_BALANCE"),
            F.col("A.LIABILITY_BALANCE"),

            *[F.col(f"A.{field}").alias(field)for field in product_columns]))

    dfs.append(comparative_result)


# =============================================================================
# FINAL UNION | COMPARATIVE
# =============================================================================

df_final_comparative = reduce(lambda df1, df2: df1.unionByName(df2),dfs)


# =============================================================================
# EXCLUDES MIGRATED CLIENTS FROM THE COMPARATIVE GROUP
# =============================================================================

df_ids_to_exclude = (df_migrated.select(F.col("CLIENT_ID").alias("RESULT_CLIENT_ID")).dropDuplicates(["RESULT_CLIENT_ID"]))

df_remove_migrated = (df_final_comparative.alias("A").join(df_ids_to_exclude.alias("B"),on="RESULT_CLIENT_ID",how="left_anti"))


# =============================================================================
# COHORT CLASSIFICATION
# =============================================================================

df_base = (
    df_remove_migrated
    .withColumn(
        "COHORT",
        F.when(F.col("ANOMES") == periods["m0"], "M0")
         .when(F.col("ANOMES") == periods["m+1"], "M1")
         .when(F.col("ANOMES") == periods["m+2"], "M2")
         .when(F.col("ANOMES") == periods["m+3"], "M3")
         .when(F.col("ANOMES") == periods["m+4"], "M4")
         .when(F.col("ANOMES") == periods["m+5"], "M5")
         .when(F.col("ANOMES") == periods["m-1"], "M-1")
         .when(F.col("ANOMES") == periods["m-2"], "M-2")
         .when(F.col("ANOMES") == periods["m-3"], "M-3")
         .when(F.col("ANOMES") == periods["m-4"], "M-4")
         .when(F.col("ANOMES") == periods["m-5"], "M-5")
         .otherwise("NULL"))
    .filter(F.col("COHORT") != "NULL")
    .withColumn("SEGMENT_CODE",
        F.when(F.col("SEGMENT_CODE").isin(["001", "002", "003", "004", "005", "006", "007", "009"]),
                F.substring(F.col("SEGMENT_CODE"), 3, 1)).otherwise(F.col("SEGMENT_CODE"))))


# =============================================================================
# COMPARATIVE AGGREGATION
# =============================================================================

comparative_grouping_columns = ["ANOMES",
                                "COHORT",
                                "SEGMENT_CODE",
                                "CLIENT_STATUS_FLAG",
                                "ACCOUNT_ACTIVITY_FLAG",
                                "PREAPPROVED_CARD_LIMIT_FLAG",
                                "IMPLEMENTED_LIMIT_FLAG",
                                "RISK_INCOME_BRACKET",
                                "RATING",
                                "CONTROL_GROUP_FLAG",
                                "PIX_WEAK_KEY",
                                "PRINCIPALITY",
                                "PIX_STRONG_KEY"]

df_client_comparative = (
    df_base
    .groupBy(*comparative_grouping_columns, "RESULT_CLIENT_ID")
    .agg(
        *[F.max(c).alias(c) for c in product_columns],
        F.sum("MOB").alias("MOB"),
        F.sum("MOL").alias("MOL"),
        F.sum("PDD").alias("PDD"),
        F.sum("ASSET_BALANCE").alias("ASSET_BALANCE"),
        F.sum("LIABILITY_BALANCE").alias("LIABILITY_BALANCE")))

comparative_aggregation_expressions = (
    [F.countDistinct("RESULT_CLIENT_ID").alias("CLIENT_COUNT")] +
    [
        F.sum("MOB").alias("MOB"),
        F.sum("MOL").alias("MOL"),
        F.sum("PDD").alias("PDD"),
        F.sum("ASSET_BALANCE").alias("ASSET_BALANCE"),
        F.sum("LIABILITY_BALANCE").alias("LIABILITY_BALANCE")] +
    [F.sum(c).alias(c) for c in product_columns])

df_aggregated_comparative = (df_client_comparative.groupBy(*comparative_grouping_columns).agg(*comparative_aggregation_expressions))

df_comparative = (
    df_aggregated_comparative
    .withColumn("CPF_COUNT", F.col("CLIENT_COUNT"))
    .withColumn("MOB_PER_CLIENT", F.col("MOB") / F.col("CLIENT_COUNT"))
    .withColumn("MOL_PER_CLIENT", F.col("MOL") / F.col("CLIENT_COUNT"))
    .withColumn("PDD_PER_CLIENT", F.col("PDD") / F.col("CLIENT_COUNT")))

ipp_columns = [(F.col(c) / F.col("CLIENT_COUNT")).alias(f"IPP_{c}") for c in product_columns]

df_comparative = df_comparative.select("*", *ipp_columns)


# =============================================================================
# FINAL WRITE | COMPARATIVE
# =============================================================================

print("SAVING COMPARATIVE RESULTS")

if len(months_to_process) > 0:

    for anomes in months_to_process:

        df_slice = df_comparative.filter(F.col("ANOMES") == anomes)

        print("=" * 70)
        print(f"SAVING ANOMES {anomes}")

        df_slice.write.mode("append").saveAsTable(TABLE_DESTINATION_COMPARATIVE)

        print(f"ANOMES {anomes} saved successfully.")

else:
    print("No new ANOMES to save.")


print("PROCESS COMPLETED SUCCESSFULLY.")


# =============================================================================
# FINAL CACHE CLEANUP
# =============================================================================

df_migrated.unpersist()
analytical_base.unpersist()
df_control_group.unpersist()
principality.unpersist()